# Middleware

Middleware provides a way to more tightly control what happens **inside the agent**.

Middleware is useful for:

- 📊 **Tracking agent behavior** with logging, analytics, and debugging.
- 🔄 **Transforming prompts, tool selection, and output formatting**.
- 🔁 **Adding retries, fallbacks, and early termination logic**.
- 🛡️ **Applying rate limits, guardrails, and PII detection**.

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automatically summarizes conversation history when approaching token limits, preserving recent messages while compressing older context.

Summarization is useful for:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages",10), # When the conversation reaches about 10 messages, trigger summarization.
            keep=("messages",4),     # Keep the most recent 4 messages and summarize the older messages.
        )
    ]
)

## So conceptually:

### Before summarization:

```text
Message 1
Message 2
Message 3
Message 4
Message 5
Message 6
Message 7
Message 8
Message 9
Message 10
```

Middleware summarizes the older messages:

```text
Summary of messages 1-6
        +
Message 7
Message 8
Message 9
Message 10
```

The exact boundary can vary because the middleware manages the history based on the configured **trigger** and **keep strategy**.

## Why do we need this?

Imagine a long conversation:

```text
User → Question
AI   → Answer
User → Question
AI   → Answer
...
User → Question
AI   → Answer
```

After hundreds of messages, sending the entire history to the model every time can consume a lot of tokens.

Summarization reduces this:

```text
100 old messages
       ↓
   Summarized
       ↓
1 summary + recent messages
```

The model still gets the important context, but the conversation history becomes much smaller.

## Overall Flow

```text
User
 ↓
Agent
 ↓
Check message count
 ↓
Reached 10 messages?
 ├── No → Continue normally
 │
 └── Yes
       ↓
SummarizationMiddleware
       ↓
Summarize older messages
       ↓
Keep latest 4 messages
       ↓
Agent continues
```

## Main Purpose

> Automatically manage long conversations by summarizing older messages while keeping the most recent messages available to the agent.

In [17]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

# The thread_id is used to identify a specific conversation.
# When using a checkpointer such as InMemorySaver(), LangChain uses the thread_id to know which conversation history to save and retrieve.

In [19]:
response =agent.invoke(
    {"messages":[
        HumanMessage(content="who is current chief Minister of TamilNadu")
    ]},
    config=config
)

In [24]:
response

{'messages': [HumanMessage(content='who is current chief Minister of TamilNadu', additional_kwargs={}, response_metadata={}, id='bd6045b2-679a-4294-a1d0-380dfef805d2'),
  AIMessage(content="\n<think>\nHere's a thinking process:\n\n1.  **Identify the User's Question**: The user is asking for the current Chief Minister of Tamil Nadu.\n2.  **Determine the Current Date/Context**: The system prompt states the current year is 2026. My knowledge cutoff is also 2026.\n3.  **Identify Key Political Facts**: \n   - Tamil Nadu is a state in India.\n   - The Chief Minister position changes based on election results.\n   - The last major state election in Tamil Nadu was in May 2021.\n   - In the 2021 Tamil Nadu Legislative Assembly election, the DMK (Dravida Munnetra Kazhagam) led alliance won.\n   - M.K. Stalin became the Chief Minister on May 7, 2021.\n   - The next scheduled state election is in 2026 (typically every 5 years).\n4.  **Consider Potential Changes by 2026**: \n   - Since my knowledge

## Token Size

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]

)

config = {"configurable": {"thread_id": "test-1"}}
# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [8]:
cities=["Chennai","Coimbatore","Bangalore"]

for city in cities:
    response=agent.invoke(
        {
            "messages":[HumanMessage(content=f"Find hotels in {city}")]
        },
        config=config
    )

    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")


Chennai: ~1357 tokens, 5 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Role:** Context Extraction Assistant\n   - **Primary Objective:** Extract highest quality/most relevant context from conversation history.\n   - **Constraint:** Free up token space by replacing history with essential context.\n   - **Structure Required:**\n     - ## SESSION INTENT\n     - ## SUMMARY\n     - ## ARTIFACTS\n     - ## NEXT STEPS\n   - **Input Conversation History:**\n     - Human: "Find hotels in Chennai"\n     - AI: Calls `search_hotels` tool with `{"city": "Chennai"}`\n     - Tool: Returns 3 hotels (Grand Hotel 5-star $350/night spa/pool/gym, City Inn 4-star $180/night business center, Budget Stay 3-star $75/night free wifi)\n     - AI: Presents the 3 hotels with details and asks if user wants more details or booking help.\n\n2.  **Evaluate Input against Requirements:**\n   - *SESSION INTENT

### Human-in-the-Loop Middleware

Pause agent execution for **human approval, editing, or rejection** of tool calls before they execute.

Human-in-the-loop is useful for:

- High-stakes operations requiring human approval (e.g., database writes or financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [58]:
agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                 "send_email_tool":{
                  "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0389479c-7dc4-4231-a39a-af9eb0fa40c7'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI have the `send_email_tool` function available.\nThe required parameters for `send_email_tool` are `recipient`, `subject`, and `body`.\nThe user provided:\n- recipient: john@test.com\n- subject: 'Hello'\n- body: 'How are you?'\n\nI have all the necessary information to make the function call. I will proceed with calling `send_email_tool`.\n", 'tool_calls': [{'id': 'yqb0e7n5f', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 374, 'total_tokens': 526, 'completion_time': 0.291412108, 'completion_tokens_details': {'reas

In [59]:
from langgraph.types import Command
# Command is used to control or resume a paused LangGraph/agent workflow.
# Step 2: Approve
if "__interrupt__" in result: 
    # interrupt is not a Python keyword.
    # it is a special key/name used by LangGraph to indicate that the agent workflow was interrupted (paused
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"} # {"type": "edit"} , {"type": "reject"} 
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Email sent to john@test.com with subject 'Hello'


In [60]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0389479c-7dc4-4231-a39a-af9eb0fa40c7'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI have the `send_email_tool` function available.\nThe required parameters for `send_email_tool` are `recipient`, `subject`, and `body`.\nThe user provided:\n- recipient: john@test.com\n- subject: 'Hello'\n- body: 'How are you?'\n\nI have all the necessary information to make the function call. I will proceed with calling `send_email_tool`.\n", 'tool_calls': [{'id': 'yqb0e7n5f', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 374, 'total_tokens': 526, 'completion_time': 0.291412108, 'completion_tokens_details': {'reas

In [61]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

In [62]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0389479c-7dc4-4231-a39a-af9eb0fa40c7'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI have the `send_email_tool` function available.\nThe required parameters for `send_email_tool` are `recipient`, `subject`, and `body`.\nThe user provided:\n- recipient: john@test.com\n- subject: 'Hello'\n- body: 'How are you?'\n\nI have all the necessary information to make the function call. I will proceed with calling `send_email_tool`.\n", 'tool_calls': [{'id': 'yqb0e7n5f', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 374, 'total_tokens': 526, 'completion_time': 0.291412108, 'completion_tokens_details': {'reas